In [4]:
import random
from collections.abc import Mapping

import httpx
import pandas as pd

In [ ]:
REFERER = "https://quote.eastmoney.com/center"

USER_AGENTS = [
    # Windows +Google Chrome
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.7922.75 Safari/537.36",
    # Iphone
    "Mozilla/5.0 (iPhone; CPU iPhone OS 18_6_2 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/18.6 Mobile/15E148 Safari/604.1",
    # Android
    "Mozilla/5.0 (Linux; Android 15; Pixel 9) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.7922.75 Mobile Safari/537.36",
]

request_headers = {
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "zh-CN,zh;q=0.9",
    "Referer":REFERER,
    "User-Agent": random.choice(USER_AGENTS)
}

{'Accept': 'application/json, text/plain, */*',
 'Accept-Language': 'zh-CN,zh;q=0.9',
 'Referer': 'https://quote.eastmoney.com/center',
 'User-Agent': 'Mozilla/5.0 (Linux; Android 15; Pixel 9) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.7922.75 Mobile Safari/537.36'}

In [10]:
client = httpx.Client(
  headers=request_headers,
  timeout=httpx.Timeout(connect=5.0, read=10.0, write=10.0, pool=10.0),
  # 设置最高连接数，防止误发大量并发，这样超量的会进行等待，超过pool timeout就报错
  limits=httpx.Limits(max_keepalive_connections=2, max_connections=2, keepalive_expiry=5.0),
  trust_env=False,
  follow_redirects=False,
)

| 参数 | 含义 | 示例 / 说明 |
| --- | --- | --- |
| `pn` | 页码 | `1` 表示第 1 页 |
| `pz` | 每页数量 | `100` 表示最多返回 100 条 |
| `fs` | 证券筛选条件 | 决定查询沪深京 A 股、ETF 等哪一类证券；编码直接复用接口示例 |
| `fields` | 返回字段列表 | 例如 `f12,f14,f2,f3` 表示代码、名称、最新价、涨跌幅 |
| `fid` | 排序字段 | `f12` 表示按股票代码排序 |
| `po` | 排序方向 | 通常固定为 `1` |
| `np` | 分页模式开关 | 通常固定为 `1` |
| `ut` | 东方财富客户端标识 | 公共固定值，通常原样保留 |
| `fltt` | 数值格式 | `1` 传原始缩放值，网页端自行格式化；`2` 直接返回小数值 |
| `invt` | 行情内部模式 | 通常保持为 `2` |

`fs` 中逗号表示合并多个证券类别；`m`、`t`、`s` 后面的数字是东方财富内部编码。

## Response 字段

| 字段 | 含义 |
| --- | --- |
| `f12` | 股票代码 |
| `f14` | 股票名称 |
| `f2` | 最新价 |
| `f3` | 涨跌幅 |
| `f4` | 涨跌额 |
| `f5` / `f6` | 成交量 / 成交额 |
| `f7` / `f8` | 振幅 / 换手率 |
| `f9` / `f10` | 动态市盈率 / 量比 |
| `f15` / `f16` | 最高价 / 最低价 |
| `f17` / `f18` | 今开 / 昨收 |
| `f20` / `f21` | 总市值 / 流通市值 |
| `f22` / `f23` | 年初至今涨跌幅 / 涨速 |
| `f24` / `f25` | 市净率 / 60 日涨跌幅 |
| `f124` / `f297` | 更新时间 / 数据日期（部分客户端使用） |

## 网页端完整 payload 解析

这次网页请求使用 JSONP 和 `fltt=1`，所以和 AkShare/efinance 的简化请求不完全相同。

### Request 参数

| 参数 | 网页中的值 | 作用 |
| --- | --- | --- |
| `cb` | `jQuery37109430227357317129_1787243717159` | JSONP 回调名；响应会包成 `cb({...})` |
| `np` | `1` | 列表型分页结果；当前 `data.diff` 是数组，部分旧代码的 `2` 可能返回不同结构 |
| `fltt` | `1` | 原始/缩放数值格式，网页端自行格式化；`2` 通常直接返回小数值 |
| `invt` | `2` | 行情接口内部模式，通常保持不变 |
| `fs` | `m:1+t:2,m:1+t:23` | 筛选沪 A 与科创板；逗号表示合并条件 |
| `fields` | `f12,f13,f14,f1,f2,f4,f3,f152` | 指定每行返回哪些 `f` 字段 |
| `fid` | `f3` | 按涨跌幅排序 |
| `pn` / `pz` | `1` / `5` | 第 1 页，每页最多 5 条 |
| `po` | `1` | 通常表示降序；`0` 通常表示升序 |
| `ut` | `fa5fd1943c7b386f172d6893dbfba10b` | 网页端公共客户端标识，不是账号认证 |
| `dect` | `1` | 网页端数字/小数格式兼容开关，复刻网页时原样保留 |
| `wbp2u` | `|0|0|0|web` | 网页客户端标记，属于内部参数 |
| `_` | `1787243717333` | 防缓存时间戳，对筛选结果没有业务影响 |

`fid` 只负责排序，`fields` 才决定返回列；`fid=f3` 与 `po=1` 通常就是按涨跌幅从高到低排列。

### `fs` 语法

| 片段 | 含义 | 示例 |
| --- | --- | --- |
| `m:<数字>` | 市场/数据源编号 | `m:0` 深 A，`m:1` 沪 A，`m:90` 板块 |
| `t:<数字>` | 证券类型或板块类型 | `t:2` 沪 A/行业板块，`t:23` 科创板，`t:80` 创业板 |
| `s:<数字>` | 更细的子类型 | `s:2048` 北证条件，`s:3` 两网及退市 |
| `b:<代码>` | 指定板块或品种集合 | `b:BKxxxx` 板块，`b:MK0354` 可转债 |
| `f:<数字>` | 特殊筛选标记 | `f:4` 风险警示，`f:8` 新股 |
| `f:!<数字>` | 排除内部标记 | `f:!50` 常用于板块和成分股列表 |

同一条件内部用 `+`（Python 字符串中也常写成空格）连接，表示同时满足；逗号表示多个条件合并。数字是东方财富内部编码，不能保证跨 endpoint 永远相同。

| 查询目标 | 常见 `fs` |
| --- | --- |
| 沪深京 A 股 | `m:0+t:6,m:0+t:80,m:1+t:2,m:1+t:23,m:0+t:81+s:2048` |
| 沪 A + 科创板 | `m:1+t:2,m:1+t:23` |
| 新股 | `m:0+f:8,m:1+f:8` |
| 风险警示板 | `m:0+f:4,m:1+f:4` |
| 两网及退市 | `m:0+s:3` |
| 行业/概念板块 | `m:90+t:2+f:!50` / `m:90+t:3+f:!50` |
| 某板块成分股 | `b:BKxxxx+f:!50` |
| ETF / LOF | `b:MK0021,b:MK0022,b:MK0023,b:MK0024,b:MK0827` / `b:MK0404,b:MK0405,b:MK0406,b:MK0407` |
| 可转债 | `b:MK0354` |
| 指数 | `b:MK0010,m:1+t:1,m:0+t:5,m:1+s:3,m:2` |
| 期货 | `m:8,m:113,m:114,m:115,m:142,m:225` |
| 美股 / 港股 / 英股 | `m:105,m:106,m:107` / `m:128` / `m:155,m:156` |

这些是 AkShare 和 efinance 源码中实际使用的常见预设，不是东方财富公开承诺的全部编码。

### Response 外层

网页响应是 `jQuery...({ ... })`；去掉 JSONP 函数名和括号后，里面才是 JSON。

| 字段 | 示例值 | 含义 |
| --- | --- | --- |
| `rc` | `0` | 返回码；`0` 通常表示成功 |
| `rt` | `6` | 服务端内部响应类型 |
| `svr` | `175646941` | 服务节点或路由标识 |
| `lt` | `1` | 服务端内部状态元数据 |
| `full` | `1` | 完整结果标记 |
| `dlmkts` | `""` | 延迟行情/市场内部元数据，本次为空 |
| `dsc` | `"0"` | 数据源或缓存内部标记 |
| `data` | `{...}` | 业务数据主体 |
| `data.total` | `2464` | 满足 `fs` 的总记录数，不是本页条数 |
| `data.diff` | `[...]` | 当前页记录；`pz=5` 时最多 5 条 |

### 本次 `fields` 对应的字段

| 字段 | 含义 | 备注 |
| --- | --- | --- |
| `f12` | 股票代码 | 例如 `600519` |
| `f13` | 市场编号 | 常用于拼接 `市场.代码`；沪市通常为 `1`，深市通常为 `0` |
| `f14` | 股票名称 | 股票简称 |
| `f1` | 内部辅助字段 | AkShare 当前行情表映射为 `_` 并忽略 |
| `f2` | 最新价 | `fltt=1` 时可能是缩放后的原始值 |
| `f4` | 涨跌额 | `fltt=1` 时按网页规则还原小数 |
| `f3` | 涨跌幅 | 本次 `fid=f3` 的排序字段 |
| `f152` | 精度/缩放辅助字段 | 与 `fltt=1` 的显示格式有关，不是普通行情指标 |

使用 `fltt=1` 时，不要把所有字段统一除以 100；价格、涨跌幅等字段要结合 `f152` 和网页格式化规则处理。

In [ ]:
target = "https://push2.eastmoney.com/api/qt/clist/get"

RemoteProtocolError: Server disconnected without sending a response.